# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Simular un escenario real de ingesta de datos ruidosos (anomalías, nulos, duplicados), ejecutar un perfilamiento estadístico para dimensionar el caos, y procesarlos a través de un flujo ETL construyendo las capas lógicas de un Data Lake (Capa *Raw* y Capa *Curated* ).



In [1]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

# Guardamos una base fresca para evitar el error de bytes corruptos en Colab
nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')

# a) Inyectamos un 5% de nulos
n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

# b) Duplicamos 50,000 registros
df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)

# c) Inyectamos outliers (valores absurdos que rompen la estadística)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

1/3 Generando datos base perfectos en memoria...
2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...
3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: 5,050,000


## Fase 1: Profiling y Capa RAW (El Diagnóstico del Caos)
Antes de transformar, debemos perfilar el archivo crudo para entender la magnitud del problema y luego guardarlo en la capa inmutable del Data Lake.

In [2]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")

# 1. Auditoría de Nulos y Duplicados
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")

# 2. Auditoría de Outliers (Fíjense en la diferencia entre el promedio y el max)
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

# 3. Guardado en Capa RAW (Inmutable)
os.makedirs('./datalake/raw', exist_ok=True)
os.makedirs('./datalake/curated', exist_ok=True)
os.makedirs('./datalake/quarantine', exist_ok=True)   # RETO 1: capa de cuarentena

print("\n--- EXTRACT & LOAD (EL) ---")
print("Guardando evidencia en la Capa RAW...")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')
print("¡Archivo RAW almacenado con éxito! Listos para limpiar.")

--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---

1. Valores Nulos por Columna:
id_transaccion         0
id_cliente             0
monto             252510
categoria              0
estado                 0
dtype: int64

2. Registros Duplicados Exactos: 49,999

3. Resumen Estadístico de 'monto':
count    4,797,490.00
mean         1,463.00
std         45,882.62
min              5.00
25%            628.91
50%          1,252.56
75%          1,876.37
max      9,999,999.99
Name: monto, dtype: object

--- EXTRACT & LOAD (EL) ---
Guardando evidencia en la Capa RAW...
¡Archivo RAW almacenado con éxito! Listos para limpiar.


## Fase 2: Transform (T) - Pipeline ETL de Limpieza y Normalización
Levantamos el dato desde RAW y ejecutamos limpieza condicionada:

1. Eliminar duplicados exactos.
2. **(Reto 2) Descarte inteligente:** si `estado == "Cancelado"` y `monto` es nulo, se elimina la fila; para los demás estados se imputa con la mediana por categoría.
3. **(Reto 1) Cuarentena:** los outliers detectados por la regla IQR **no se borran**: se guardan en Parquet en `./datalake/quarantine`.
4. Normalización Min-Max para Machine Learning.

In [3]:
print("Iniciando Transformación...\n")
filas_raw = len(df_ingesta)

# 1. Eliminación de Duplicados
df_limpio = df_ingesta.drop_duplicates().copy()
n_duplicados = filas_raw - len(df_limpio)
print(f"Duplicados eliminados: {n_duplicados:,}")

# 2. RETO 2 - Descarte inteligente de nulos en 'monto'
#    a) Cancelado + monto nulo -> se elimina la fila (no aporta valor predictivo)
mask_descarte = (df_limpio['estado'] == 'Cancelado') & (df_limpio['monto'].isnull())
n_descartados = int(mask_descarte.sum())
df_limpio = df_limpio[~mask_descarte].copy()
print(f"Filas 'Cancelado' con monto nulo descartadas: {n_descartados:,}")

#    b) Resto de estados (Completado, Pendiente, Reembolsado) -> imputación con mediana por categoría
n_a_imputar = int(df_limpio['monto'].isnull().sum())
medianas = df_limpio.groupby('categoria')['monto'].transform('median')
df_limpio['monto'] = df_limpio['monto'].fillna(medianas)
print(f"Nulos imputados con mediana por categoría: {n_a_imputar:,}")
print(f"Nulos restantes en monto: {df_limpio['monto'].isnull().sum()}")

# 3. RETO 1 - Outliers (Regla IQR) -> CUARENTENA en lugar de borrado
Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
print(f"Límites IQR: [{limite_inferior:,.2f}, {limite_superior:,.2f}]")

mask_outlier = (df_limpio['monto'] < limite_inferior) | (df_limpio['monto'] > limite_superior)
df_cuarentena = df_limpio[mask_outlier].copy()
df_cuarentena['motivo_cuarentena'] = np.where(
    df_cuarentena['monto'] > limite_superior, 'outlier_sobre_limite_superior', 'outlier_bajo_limite_inferior')

os.makedirs('./datalake/quarantine', exist_ok=True)
df_cuarentena.to_parquet('./datalake/quarantine/outliers_iqr.parquet')
print(f"Outliers enviados a CUARENTENA (no eliminados): {len(df_cuarentena):,}")

df_limpio = df_limpio[~mask_outlier].copy()

# 4. Normalización Min-Max (Escala de 0 a 1)
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()
df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
print("Normalización Min-Max completada.")

Iniciando Transformación...

Duplicados eliminados: 49,999
Filas 'Cancelado' con monto nulo descartadas: 62,513
Nulos imputados con mediana por categoría: 187,479
Nulos restantes en monto: 0
Límites IQR: [-1,144.36, 3,649.68]
Outliers enviados a CUARENTENA (no eliminados): 101
Normalización Min-Max completada.


## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el dato limpio en formato columnar en la capa *Curated*. Finalmente, comparamos la estadística del dato curado frente al desastre original.

In [4]:
# 1. Guardamos en la capa analítica
df_limpio.to_parquet('./datalake/curated/transacciones_model_ready.parquet')
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 2. Demostración del dato limpio (El "Después")
print("--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---")
# Comparamos la estadística final para demostrar que la media volvió a la normalidad
print(df_limpio[['monto', 'monto_escalado']].describe().apply(lambda col: col.map(lambda x: f"{x:,.2f}")))

print("\n--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'monto', 'monto_escalado']].head(5))

print("\n--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---")
print(f"RAW (Sucio): {os.path.getsize('./datalake/raw/ingesta_transacciones_sucio.parquet') / (1024*1024):.2f} MB")
print(f"CURATED (Limpio): {os.path.getsize('./datalake/curated/transacciones_model_ready.parquet') / (1024*1024):.2f} MB")

# 3. RETO 1 - Verificación de la capa de cuarentena (lectura desde disco)
print("\n--- VERIFICACIÓN DE LA CAPA QUARANTINE ---")
df_q = pd.read_parquet('./datalake/quarantine/outliers_iqr.parquet')
print(f"Registros en cuarentena: {len(df_q):,}")
print(df_q['motivo_cuarentena'].value_counts())
display(df_q.head(5))

# 4. Reconciliación: ninguna fila debe perderse sin explicación
print("\n--- RECONCILIACIÓN DE FILAS (RAW = CURATED + QUARANTINE + DESCARTES) ---")
print(f"RAW (ingesta)                        : {filas_raw:,}")
print(f"  - Duplicados eliminados            : {n_duplicados:,}")
print(f"  - Cancelado + monto nulo (Reto 2)  : {n_descartados:,}")
print(f"  - En CUARENTENA (Reto 1)           : {len(df_cuarentena):,}")
print(f"  = CURATED                          : {len(df_limpio):,}")
assert filas_raw == len(df_limpio) + len(df_cuarentena) + n_duplicados + n_descartados
print("Reconciliación OK: todas las filas están contabilizadas.")

# 5. Estructura final del Data Lake (las tres carpetas)
print("\n--- ESTRUCTURA DEL DATA LAKE ---")
for carpeta in ['raw', 'curated', 'quarantine']:
    ruta = f'./datalake/{carpeta}'
    print(f"{ruta}/")
    for archivo in os.listdir(ruta):
        peso = os.path.getsize(os.path.join(ruta, archivo)) / (1024 * 1024)
        print(f"    {archivo}  ({peso:.2f} MB)")

Datos escritos en la Capa CURATED exitosamente.

--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---
              monto monto_escalado
count  4,937,387.00   4,937,387.00
mean       1,252.50           0.50
std          706.46           0.28
min            5.00           0.00
25%          653.39           0.26
50%        1,252.53           0.50
75%        1,851.88           0.74
max        2,500.00           1.00

--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---


,id_transaccion,categoria,monto,monto_escalado
101,101,Juguetes,1015.580233,0.405042
102,102,Alimentos,1837.507104,0.734472
103,103,Electronica,997.149883,0.397655
104,104,Ropa,2021.055379,0.808038
105,105,Juguetes,1251.907721,0.499762



--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---
RAW (Sucio): 74.41 MB
CURATED (Limpio): 133.05 MB

--- VERIFICACIÓN DE LA CAPA QUARANTINE ---
Registros en cuarentena: 101
motivo_cuarentena
outlier_sobre_limite_superior    101
Name: count, dtype: int64


,id_transaccion,id_cliente,monto,categoria,estado,motivo_cuarentena
0,0,15796,9999999.99,Alimentos,Cancelado,outlier_sobre_limite_superior
1,1,861,9999999.99,Juguetes,Cancelado,outlier_sobre_limite_superior
2,2,76821,9999999.99,Ropa,Pendiente,outlier_sobre_limite_superior
3,3,54887,9999999.99,Alimentos,Reembolsado,outlier_sobre_limite_superior
4,4,6266,9999999.99,Ropa,Pendiente,outlier_sobre_limite_superior



--- RECONCILIACIÓN DE FILAS (RAW = CURATED + QUARANTINE + DESCARTES) ---
RAW (ingesta)                        : 5,050,000
  - Duplicados eliminados            : 49,999
  - Cancelado + monto nulo (Reto 2)  : 62,513
  - En CUARENTENA (Reto 1)           : 101
  = CURATED                          : 4,937,387
Reconciliación OK: todas las filas están contabilizadas.

--- ESTRUCTURA DEL DATA LAKE ---
./datalake/raw/
    ingesta_transacciones_sucio.parquet  (74.41 MB)
./datalake/curated/
    transacciones_model_ready.parquet  (133.05 MB)
./datalake/quarantine/
    outliers_iqr.parquet  (0.01 MB)
